In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# fetch dataset
car_evaluation = fetch_ucirepo(id=19)

#Data scource : https://archive.ics.uci.edu/dataset/19/car+evaluation
# data (as pandas dataframes)
data = car_evaluation.data.features
data["target"] = car_evaluation.data.targets

In [ ]:
data.head()

,buying,maint,doors,persons,lug_boot,safety,target
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   buying     1728 non-null   object
 1   maint      1728 non-null   object
 2   doors      1728 non-null   object
 3   persons    1728 non-null   object
 4   lug_boot   1728 non-null   object
 5   safety     1728 non-null   object
 6   car_class  1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB


In [ ]:
import patsy
from patsy.contrasts import Sum, Diff

contrast_dict = {
    "buying": Diff(),
    "maint": Diff(),
    "doors": Sum(),
    "persons": Sum(),
    "lug_boot": Sum(),
    "safety": Sum(),
    "car_class": Sum()
}

# Build formula dynamically
feature_columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]
formula_components = []
for col in feature_columns:
    contrast_class_name = contrast_dict[col].__class__.__name__
    formula_components.append(f"C({col}, {contrast_class_name})")

rhs_formula = " + ".join(formula_components)
full_formula = f"car_class ~ {rhs_formula}"

# Build design matrices
y, X = patsy.dmatrices(full_formula, data, return_type="dataframe", NA_action="drop")

# Convert target to numeric labels
y = y.idxmax(axis=1).astype("category").cat.codes

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CarANN(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32, hidden3=16, output_dim=4):
        super(CarANN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, hidden3)
        self.out = nn.Linear(hidden3, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.out(x)

model = CarANN(input_dim=X_tensor.shape[1])

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.2219
Epoch 10, Loss: 0.1086
Epoch 20, Loss: 0.0065
Epoch 30, Loss: 0.0037
Epoch 40, Loss: 0.0076


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

with torch.no_grad():
    outputs = model(X_test)
    preds = torch.argmax(outputs, dim=1)

print("Test Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=data["car_class"].unique()))

Test Accuracy: 0.9971098265895953
              precision    recall  f1-score   support

       unacc       1.00      0.99      0.99        77
         acc       0.93      1.00      0.97        14
       vgood       1.00      1.00      1.00       242
        good       1.00      1.00      1.00        13

    accuracy                           1.00       346
   macro avg       0.98      1.00      0.99       346
weighted avg       1.00      1.00      1.00       346

